Okay, here's the SVM classification code for the EEG Motor Movement/Imagery Dataset, broken down into sections with Markdown headings and explanations, perfectly suited for a Jupyter Notebook.

-----

# EEG Motor Movement/Imagery Classification with SVM

This notebook demonstrates how to load, preprocess, extract features from, and classify EEG data from the PhysioNet Motor Movement/Imagery Dataset using a Support Vector Machine (SVM) classifier.

## 1\. Setup and Configuration

First, we import the necessary libraries and define the root directory where your dataset is stored. We also set up the mapping for the different experimental events to the target classes for our classification task.

In [1]:
# --- 1.1. Import Libraries ---
import mne
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from mne.time_frequency import psd_array_multitaper

# --- 1.2. Define Data Path ---
# IMPORTANT: Change this to the actual path where your dataset is extracted
data_root_dir = 'C:/ESC5Work/ProjectM5/files'

# --- 1.3. Define Event & Target Class Mappings ---
# Custom event labels based on run type (to differentiate between Left/Both Fists, Real/Imagine)
CUSTOM_EVENT_LABELS = {
    'T0': 'Rest',

    # Runs 3, 4, 7, 8, 11, 12: Left/Right Fist Tasks
    'R03_T1': 'Left_Fist_Real',
    'R04_T1': 'Left_Fist_Imagine',
    'R07_T1': 'Left_Fist_Real',
    'R08_T1': 'Left_Fist_Imagine',
    'R11_T1': 'Left_Fist_Real',
    'R12_T1': 'Left_Fist_Imagine',

    'R03_T2': 'Right_Fist_Real',
    'R04_T2': 'Right_Fist_Imagine',
    'R07_T2': 'Right_Fist_Real',
    'R08_T2': 'Right_Fist_Imagine',
    'R11_T2': 'Right_Fist_Real',
    'R12_T2': 'Right_Fist_Imagine',

    # Runs 5, 6, 9, 10, 13, 14: Both Fists/Both Feet Tasks
    'R05_T1': 'Both_Fists_Real',
    'R06_T1': 'Both_Fists_Imagine',
    'R09_T1': 'Both_Fists_Real',
    'R10_T1': 'Both_Fists_Imagine',
    'R13_T1': 'Both_Fists_Real',
    'R14_T1': 'Both_Fists_Imagine',

    'R05_T2': 'Both_Feet_Real',
    'R06_T2': 'Both_Feet_Imagine',
    'R09_T2': 'Both_Feet_Real',
    'R10_T2': 'Both_Feet_Imagine',
    'R13_T2': 'Both_Feet_Real',
    'R14_T2': 'Both_Feet_Imagine',
}

# Define the final classes for our classifier. We group real and imagined actions.
TARGET_CLASSES = {
    'Rest': 0,
    'Left_Fist': 1,
    'Right_Fist': 2,
    'Both_Fists': 3,
    'Both_Feet': 4
}

# Lists to store features and labels from all processed epochs
all_epochs_features = []
all_epochs_labels = []

print("Setup complete. Starting data processing...")

Setup complete. Starting data processing...


## 2\. Data Loading, Preprocessing, and Feature Extraction Loop

This is the core loop that iterates through each subject and each experimental run. For every EDF file, it performs the following steps:

1.  **Loads Raw EEG Data:** Reads the `.edf` file into an MNE `Raw` object.
2.  **Preprocessing:**
      * Applies a Common Average Reference (CAR) to reduce common noise across electrodes.
      * Applies a band-pass filter (0.5 to 30 Hz) to focus on relevant EEG frequencies and remove artifacts.
3.  **Event Extraction & Epoching:**
      * Identifies `T0`, `T1`, `T2` events from the annotations within the `edf` file.
      * Segments the continuous `Raw` data into `Epochs`, using a window from -0.2 seconds before the event to 2.5 seconds after. Baseline correction is applied using the pre-event period.
      * Epochs overlapping with 'BAD' annotations are automatically rejected.
4.  **Label Mapping:**
      * Converts the generic `T0`/`T1`/`T2` event IDs into more specific action/imagery labels (e.g., 'Left\_Fist\_Real', 'Both\_Feet\_Imagine') based on the `run_id`.
      * Then, maps these specific labels to our final `TARGET_CLASSES` (e.g., 'Left\_Fist', 'Both\_Fists') to simplify the classification problem.
5.  **Feature Extraction:**
      * Calculates the Power Spectral Density (PSD) for each epoch using the multitaper method. PSDs represent the power distribution across different frequencies, which are excellent features for EEG classification.
      * Flattens the PSD features into a 1D array for each epoch, suitable for input to the SVM.

<!-- end list -->

In [2]:
%%capture

# Adjust the range for 'subject_id' if you want to process a subset of subjects
# (e.g., range(1, 11) for S001-S010 for a quicker test)
for subject_id in range(1, 110):
    s_id = f'S{subject_id:03d}'
    subject_folder = os.path.join(data_root_dir, s_id)

    if not os.path.exists(subject_folder):
        print(f"Warning: Folder not found for {s_id}. Skipping.")
        continue

    # All 14 runs per subject
    for run_id in range(1, 15):
        r_id = f'R{run_id:02d}'
        edf_file_name = f'{s_id}{r_id}.edf'
        edf_file_path = os.path.join(subject_folder, edf_file_name)

        if not os.path.exists(edf_file_path):
            # print(f"Warning: File not found: {edf_file_path}. Skipping.")
            continue

        print(f"Processing: {edf_file_path}")

        try:
            # Load the EDF file
            raw = mne.io.read_raw_edf(edf_file_path, preload=True, verbose=False)

            # Preprocessing: Common Average Reference and Band-pass filter
            raw.set_eeg_reference('average', projection=True).apply_proj()
            raw.filter(0.5, 30, fir_design='firwin', verbose=False)

            # Extract events from annotations
            events, event_id_from_mne = mne.events_from_annotations(raw, verbose=False)

            # Define epoch time window: -0.2s before event to 2.5s after event
            tmin_epoch = -0.2
            tmax_epoch = 2.5

            # Create Epochs
            epochs = mne.Epochs(
                raw,
                events,
                event_id=event_id_from_mne, # MNE's auto-detected T0, T1, T2 IDs
                tmin=tmin_epoch,
                tmax=tmax_epoch,
                baseline=(None, 0), # Baseline correction from tmin_epoch to event onset
                preload=True,
                reject_by_annotation=True,
                verbose=False
            )

            # Prepare labels for the current run's epochs based on TARGET_CLASSES
            current_run_epochs_labels = []
            reverse_mne_id_map = {v: k for k, v in event_id_from_mne.items()} # MNE int ID to T0/T1/T2 string

            for epoch_idx in range(len(epochs)):
                mne_int_label = epochs.events[epoch_idx, 2]
                base_label_str = reverse_mne_id_map.get(mne_int_label) # e.g., 'T0', 'T1', 'T2'

                # Construct the key for CUSTOM_EVENT_LABELS
                full_label_key = f'R{run_id:02d}_{base_label_str}' if base_label_str != 'T0' else base_label_str
                specific_label_string = CUSTOM_EVENT_LABELS.get(full_label_key)

                # Map to our final TARGET_CLASSES integers
                if specific_label_string:
                    if 'Left_Fist' in specific_label_string:
                        current_run_epochs_labels.append(TARGET_CLASSES['Left_Fist'])
                    elif 'Right_Fist' in specific_label_string:
                        current_run_epochs_labels.append(TARGET_CLASSES['Right_Fist'])
                    elif 'Both_Fists' in specific_label_string:
                        current_run_epochs_labels.append(TARGET_CLASSES['Both_Fists'])
                    elif 'Both_Feet' in specific_label_string:
                        current_run_epochs_labels.append(TARGET_CLASSES['Both_Feet'])
                    elif specific_label_string == 'Rest':
                        current_run_epochs_labels.append(TARGET_CLASSES['Rest'])
                    else:
                        current_run_epochs_labels.append(None) # Mark for removal
                else:
                    current_run_epochs_labels.append(None) # Mark for removal

            # Filter out epochs that didn't map to a target class
            valid_indices = [i for i, label in enumerate(current_run_epochs_labels) if label is not None]

            if not valid_indices:
                del raw, epochs
                continue

            epochs_selected = epochs[valid_indices]
            labels_selected = np.array([current_run_epochs_labels[i] for i in valid_indices])

            if len(epochs_selected) == 0:
                del raw, epochs_selected
                continue

            # Feature Extraction: Power Spectral Density (PSD)
            # PSDs are calculated across all channels and the specified frequency range
            psds, freqs = psd_array_multitaper(epochs_selected.get_data(), raw.info['sfreq'],
                                               fmin=0.5, fmax=30, verbose=False)

            # Flatten PSDs: (n_epochs, n_channels, n_frequencies) -> (n_epochs, n_channels * n_frequencies)
            features_for_run = psds.reshape(psds.shape[0], -1)

            # Store features and labels
            all_epochs_features.append(features_for_run)
            
            all_epochs_labels.append(labels_selected)

            # Clean up memory
            del raw, epochs, epochs_selected

        except Exception as e:
            print(f"Error processing {edf_file_path}: {e}")
            continue

print("\n--- Data Loading, Preprocessing, and Feature Extraction Complete ---")

Processing: C:/ESC5Work/ProjectM5/files\S001\S001R01.edf
EEG channel type selected for re-referencing
Adding average EEG reference projection.
1 projection items deactivated
Average reference projection was added, but has not been applied yet. Use the apply_proj method to apply it.
Created an SSP operator (subspace dimension = 1)
1 projection items activated
SSP projectors applied...
Processing: C:/ESC5Work/ProjectM5/files\S001\S001R02.edf
EEG channel type selected for re-referencing
Adding average EEG reference projection.
1 projection items deactivated
Average reference projection was added, but has not been applied yet. Use the apply_proj method to apply it.
Created an SSP operator (subspace dimension = 1)
1 projection items activated
SSP projectors applied...
Processing: C:/ESC5Work/ProjectM5/files\S001\S001R03.edf
EEG channel type selected for re-referencing


C:\Users\scpwk\AppData\Local\Temp\ipykernel_9992\3163159924.py:39: RuntimeWarning: All epochs were dropped!
You might need to alter reject/flat-criteria or drop bad channels to avoid this. You can use Epochs.plot_drop_log() to see which channels are responsible for the dropping of epochs.
  epochs = mne.Epochs(
C:\Users\scpwk\AppData\Local\Temp\ipykernel_9992\3163159924.py:39: RuntimeWarning: All epochs were dropped!
You might need to alter reject/flat-criteria or drop bad channels to avoid this. You can use Epochs.plot_drop_log() to see which channels are responsible for the dropping of epochs.
  epochs = mne.Epochs(


Adding average EEG reference projection.
1 projection items deactivated
Average reference projection was added, but has not been applied yet. Use the apply_proj method to apply it.
Created an SSP operator (subspace dimension = 1)
1 projection items activated
SSP projectors applied...
Processing: C:/ESC5Work/ProjectM5/files\S001\S001R04.edf
EEG channel type selected for re-referencing
Adding average EEG reference projection.
1 projection items deactivated
Average reference projection was added, but has not been applied yet. Use the apply_proj method to apply it.
Created an SSP operator (subspace dimension = 1)
1 projection items activated
SSP projectors applied...
Processing: C:/ESC5Work/ProjectM5/files\S001\S001R05.edf
EEG channel type selected for re-referencing
Adding average EEG reference projection.
1 projection items deactivated
Average reference projection was added, but has not been applied yet. Use the apply_proj method to apply it.
Created an SSP operator (subspace dimension =

C:\Users\scpwk\AppData\Local\Temp\ipykernel_9992\3163159924.py:39: RuntimeWarning: All epochs were dropped!
You might need to alter reject/flat-criteria or drop bad channels to avoid this. You can use Epochs.plot_drop_log() to see which channels are responsible for the dropping of epochs.
  epochs = mne.Epochs(
C:\Users\scpwk\AppData\Local\Temp\ipykernel_9992\3163159924.py:39: RuntimeWarning: All epochs were dropped!
You might need to alter reject/flat-criteria or drop bad channels to avoid this. You can use Epochs.plot_drop_log() to see which channels are responsible for the dropping of epochs.
  epochs = mne.Epochs(


Adding average EEG reference projection.
1 projection items deactivated
Average reference projection was added, but has not been applied yet. Use the apply_proj method to apply it.
Created an SSP operator (subspace dimension = 1)
1 projection items activated
SSP projectors applied...
Processing: C:/ESC5Work/ProjectM5/files\S002\S002R04.edf
EEG channel type selected for re-referencing
Adding average EEG reference projection.
1 projection items deactivated
Average reference projection was added, but has not been applied yet. Use the apply_proj method to apply it.
Created an SSP operator (subspace dimension = 1)
1 projection items activated
SSP projectors applied...
Processing: C:/ESC5Work/ProjectM5/files\S002\S002R05.edf
EEG channel type selected for re-referencing
Adding average EEG reference projection.
1 projection items deactivated
Average reference projection was added, but has not been applied yet. Use the apply_proj method to apply it.
Created an SSP operator (subspace dimension =

C:\Users\scpwk\AppData\Local\Temp\ipykernel_9992\3163159924.py:39: RuntimeWarning: All epochs were dropped!
You might need to alter reject/flat-criteria or drop bad channels to avoid this. You can use Epochs.plot_drop_log() to see which channels are responsible for the dropping of epochs.
  epochs = mne.Epochs(
C:\Users\scpwk\AppData\Local\Temp\ipykernel_9992\3163159924.py:39: RuntimeWarning: All epochs were dropped!
You might need to alter reject/flat-criteria or drop bad channels to avoid this. You can use Epochs.plot_drop_log() to see which channels are responsible for the dropping of epochs.
  epochs = mne.Epochs(


... (247 outputs left out)

C:\Users\scpwk\AppData\Local\Temp\ipykernel_9992\3163159924.py:39: RuntimeWarning: All epochs were dropped!
You might need to alter reject/flat-criteria or drop bad channels to avoid this. You can use Epochs.plot_drop_log() to see which channels are responsible for the dropping of epochs.
  epochs = mne.Epochs(
C:\Users\scpwk\AppData\Local\Temp\ipykernel_9992\3163159924.py:39: RuntimeWarning: All epochs were dropped!
You might need to alter reject/flat-criteria or drop bad channels to avoid this. You can use Epochs.plot_drop_log() to see which channels are responsible for the dropping of epochs.
  epochs = mne.Epochs(


EEG channel type selected for re-referencing
Adding average EEG reference projection.
1 projection items deactivated
Average reference projection was added, but has not been applied yet. Use the apply_proj method to apply it.
Created an SSP operator (subspace dimension = 1)
1 projection items activated
SSP projectors applied...
Processing: C:/ESC5Work/ProjectM5/files\S108\S108R04.edf
EEG channel type selected for re-referencing
Adding average EEG reference projection.
1 projection items deactivated
Average reference projection was added, but has not been applied yet. Use the apply_proj method to apply it.
Created an SSP operator (subspace dimension = 1)
1 projection items activated
SSP projectors applied...
Processing: C:/ESC5Work/ProjectM5/files\S108\S108R05.edf
EEG channel type selected for re-referencing
Adding average EEG reference projection.
1 projection items deactivated
Average reference projection was added, but has not been applied yet. Use the apply_proj method to apply it.


C:\Users\scpwk\AppData\Local\Temp\ipykernel_9992\3163159924.py:39: RuntimeWarning: All epochs were dropped!
You might need to alter reject/flat-criteria or drop bad channels to avoid this. You can use Epochs.plot_drop_log() to see which channels are responsible for the dropping of epochs.
  epochs = mne.Epochs(
C:\Users\scpwk\AppData\Local\Temp\ipykernel_9992\3163159924.py:39: RuntimeWarning: All epochs were dropped!
You might need to alter reject/flat-criteria or drop bad channels to avoid this. You can use Epochs.plot_drop_log() to see which channels are responsible for the dropping of epochs.
  epochs = mne.Epochs(


Adding average EEG reference projection.
1 projection items deactivated
Average reference projection was added, but has not been applied yet. Use the apply_proj method to apply it.
Created an SSP operator (subspace dimension = 1)
1 projection items activated
SSP projectors applied...
Processing: C:/ESC5Work/ProjectM5/files\S109\S109R04.edf
EEG channel type selected for re-referencing
Adding average EEG reference projection.
1 projection items deactivated
Average reference projection was added, but has not been applied yet. Use the apply_proj method to apply it.
Created an SSP operator (subspace dimension = 1)
1 projection items activated
SSP projectors applied...
Processing: C:/ESC5Work/ProjectM5/files\S109\S109R05.edf
EEG channel type selected for re-referencing
Adding average EEG reference projection.
1 projection items deactivated
Average reference projection was added, but has not been applied yet. Use the apply_proj method to apply it.
Created an SSP operator (subspace dimension =

## 3\. Combine All Data

After processing all files, we concatenate the collected features and labels into single NumPy arrays. These will form our complete dataset `X` (features) and `y` (labels).

In [3]:
X = np.concatenate(all_epochs_features, axis=0) if all_epochs_features else np.array([])
y = np.concatenate(all_epochs_labels, axis=0) if all_epochs_labels else np.array([])

print(f"Total shape of X (features): {X.shape}")
print(f"Total shape of y (labels): {y.shape}")

print("\nFinal Label Distribution:")
unique_labels, counts = np.unique(y, return_counts=True)
reverse_target_map = {v: k for k, v in TARGET_CLASSES.items()}
for label_id, count in zip(unique_labels, counts):
    print(f"  {reverse_target_map.get(label_id, 'Unknown')} (ID {label_id}): {count} epochs")

Total shape of X (features): (38041, 5120)
Total shape of y (labels): (38041,)

Final Label Distribution:
  Rest (ID 0): 18367 epochs
  Left_Fist (ID 1): 4951 epochs
  Right_Fist (ID 2): 4894 epochs
  Both_Fists (ID 3): 4905 epochs
  Both_Feet (ID 4): 4924 epochs


## 4\. Data Splitting

We split the combined dataset into training and testing sets. A `stratify` parameter is used to ensure that the proportion of each class is maintained in both the training and testing sets, which is good practice for classification tasks, especially with potentially imbalanced classes.

In [4]:
# Using a simple random split for demonstration.
# For more robust evaluation in BCI, consider subject-wise splitting.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training data shape: {X_train.shape}, labels: {y_train.shape}")
print(f"Testing data shape: {X_test.shape}, labels: {y_test.shape}")

Training data shape: (30432, 5120), labels: (30432,)
Testing data shape: (7609, 5120), labels: (7609,)


## 5\. Feature Scaling

SVMs are sensitive to the scale of input features. We use `StandardScaler` to transform the features to have zero mean and unit variance. This step is applied independently to the training and testing sets (fitting on training, transforming both).

In [5]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nFeatures scaled.")


Features scaled.


## 6\. SVM Model Training

Here, we initialize and train the Support Vector Machine classifier. Key parameters for the SVM include `C` (regularization strength) and `kernel` (the function used to transform input data into a higher-dimensional feature space). The 'rbf' (Radial Basis Function) kernel is a popular choice for its flexibility.

In [6]:
print("\n--- Training Support Vector Machine (SVM) ---")

svm_classifier = SVC(
    C=1.0,         # Regularization parameter. Adjust for complexity/overfitting.
    kernel='rbf',  # Radial Basis Function kernel for non-linear decision boundaries.
    gamma='scale', # Kernel coefficient. 'scale' uses 1 / (n_features * X.var()).
    random_state=42,
    verbose=True  # Set to True to see training progress (for very large datasets)
)

svm_classifier.fit(X_train_scaled, y_train)

print("SVM training complete.")


--- Training Support Vector Machine (SVM) ---
[LibSVM]SVM training complete.


## 7\. Model Evaluation

Finally, we evaluate the trained SVM model on the unseen test data. We'll use the classification report (precision, recall, f1-score per class), overall accuracy, and a confusion matrix to assess performance.

In [7]:
y_pred = svm_classifier.predict(X_test_scaled)

print("\n--- SVM Evaluation ---")
print("Classification Report:")
# Ensure target_names are in the correct order for the report
report_target_names = [reverse_target_map[i] for i in sorted(np.unique(y))]
print(classification_report(y_test, y_pred, target_names=report_target_names, zero_division=0)) # zero_division=0 to avoid warning for classes with no predictions

print("\nAccuracy Score:")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")

print("\nConfusion Matrix:")
conf_matrix = confusion_matrix(y_test, y_pred)
conf_df = pd.DataFrame(conf_matrix,
                       index=report_target_names,
                       columns=report_target_names)
print(conf_df)

print("\n--- End of Notebook ---")
# You can now interpret the results of your SVM classifier.
# If performance is not satisfactory, consider hyperparameter tuning (e.g., GridSearchCV for C and gamma),
# exploring different feature extraction methods (like Common Spatial Patterns (CSP) for MI data),
# or collecting more data.


--- SVM Evaluation ---
Classification Report:
              precision    recall  f1-score   support

        Rest       0.49      1.00      0.66      3674
   Left_Fist       0.70      0.02      0.03       990
  Right_Fist       0.55      0.02      0.04       979
  Both_Fists       0.71      0.03      0.06       981
   Both_Feet       0.82      0.05      0.09       985

    accuracy                           0.50      7609
   macro avg       0.65      0.22      0.18      7609
weighted avg       0.60      0.50      0.35      7609


Accuracy Score:
Accuracy: 0.4966

Confusion Matrix:
            Rest  Left_Fist  Right_Fist  Both_Fists  Both_Feet
Rest        3666          2           2           0          4
Left_Fist    958         16           6           5          5
Right_Fist   956          2          18           3          0
Both_Fists   944          0           5          30          2
Both_Feet    927          3           2           4         49

--- End of Notebook ---
